# STAT5106 Final Project — Part B: Data Cleaning & Feature Engineering

**Research question:** Which product and reviewer characteristics are associated with a five-star skincare review on Sephora?

**Input:** `merged_reviews.parquet` from Part A (1,093,970 reviews × 46 columns). Part A already combined the 5 review files, removed 441 exact duplicates, filled skin attributes for authors who reported them consistently in other reviews, and matched every review to the product table.

**Outputs**

| File | Purpose |
|---|---|
| `data/processed/analysis_ready.parquet` | One row per review, outcomes + cleaned/encoded predictors. Used by Parts C–F. |
| `data/processed/review_text_lookup.parquet` | `review_id` + title/text, kept apart (leakage + file size). Only for the optional topic model. |
| `outputs/cleaning_summary.csv` | Every step with rows before/after and the reason. |
| `outputs/feature_dictionary.csv` | Every column in `analysis_ready.parquet`: meaning, source, role, levels. |
| `outputs/feature_selection.csv` | Every raw column: kept or excluded, and why. |
| `outputs/figures/*.png` | Figures for the Preprocessing slide. |

**Steps**
1. Load and sanity-check Part A's file
2. Investigate repeated author–product pairs
3. Build outcome variables and check class balance
4. Feature selection
5. Missing values
6. Rare categories (merge by meaning)
7. Price segments and log price
8. Highlights indicators
9. Time variables
10. Categorical encoding
11. Data-leakage check
12. Model-readiness checks
13. Assemble, save and validate

In [ ]:
# Find only this project (never search the user's home directory).
import sys
from pathlib import Path
_PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "project_paths.py").is_file()), None)
if _PROJECT_ROOT is None:
    raise FileNotFoundError("Open the notebook within the project folder.")
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))
from project_paths import ROOT, RESULTS_DIR, ANALYSIS_PATH, RAW_DIR, PROCESSED_DIR, require_analysis, require_merged, output_dir

from pathlib import Path
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 120)

SEED = 5106

BASE_DIR = ROOT
INPUT_PATH = require_merged()
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = output_dir("B")
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Chart style (one blue for data, grey for context)
BLUE, ORANGE, GREY, INK, INK2 = "#2a78d6", "#eb6834", "#b8b7b1", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.edgecolor": GREY, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#ececea", "grid.linewidth": 0.8, "axes.axisbelow": True,
})

# Cleaning log helper: every step records rows before/after and why
cleaning_log = []
def log_step(step, action, reason, rows_before, rows_after, detail=""):
    cleaning_log.append({
        "step": step, "action": action, "reason": reason,
        "rows_before": rows_before, "rows_after": rows_after,
        "rows_removed": rows_before - rows_after, "detail": detail,
    })

print("Input:", INPUT_PATH)
print("pandas", pd.__version__, "| numpy", np.__version__)

## Step 1. Load and sanity-check Part A's file

Before cleaning, confirm that Part A's output is what we expect: valid ratings, positive prices, a single primary category (Skincare), every review matched to a product, and no exact duplicates left. Part A's merge also created three copies of product columns (`product_product_name`, `product_brand_name`, `product_price_usd`); if they are identical to the review-table versions we drop them.

In [ ]:
df = pd.read_parquet(INPUT_PATH)
n0 = len(df)
print("Shape:", df.shape)

checks = {
    "rating within 1-5": df["rating"].isin([1, 2, 3, 4, 5]).all(),
    "rating has no missing": df["rating"].notna().all(),
    "price_usd > 0 and not missing": (df["price_usd"] > 0).all() and df["price_usd"].notna().all(),
    "primary_category is only Skincare": set(df["primary_category"].unique()) == {"Skincare"},
    "all reviews matched to products": (df["product_match_status"] == "matched").all(),
    "product_id never missing": df["product_id"].notna().all(),
    "author_id never missing": df["author_id"].notna().all(),
    "submission_time never missing": df["submission_time"].notna().all(),
}
for k, v in checks.items():
    print(f"{'OK ' if v else 'FAIL'} {k}")
assert all(checks.values())

print("Review dates:", df["submission_time"].min().date(), "to", df["submission_time"].max().date())
print("Products:", df["product_id"].nunique(), "| Authors:", df["author_id"].nunique(), "| Brands:", df["brand_name"].nunique())

# Exact duplicates should already be gone (Part A removed 441)
dup_cols = [c for c in df.columns if c not in
            ["helpfulness", "total_feedback_count", "total_neg_feedback_count",
             "total_pos_feedback_count", "source_file"]]
n_exact_dup = int(df.duplicated(subset=dup_cols).sum())
print("Exact duplicate reviews remaining:", n_exact_dup)
# Part A filled skin attributes AFTER de-duplicating, which can make two rows identical again
if n_exact_dup:
    display(df[df.duplicated(subset=dup_cols, keep=False)][["author_id", "product_id", "rating", "submission_time", "skin_type", "review_title"]])
    n_before = len(df)
    df = df.drop_duplicates(subset=dup_cols, keep="first").reset_index(drop=True)
    log_step("1. Load & validate", "Removed exact duplicates that re-appeared after Part A's attribute filling",
             "Rows identical on every review field", n_before, len(df), f"{n_before - len(df)} rows removed")
assert not df.duplicated(subset=dup_cols).any()

# Redundant columns created by the merge
redundant = {"product_product_name": "product_name",
             "product_brand_name": "brand_name",
             "product_price_usd": "price_usd"}
for extra, base in redundant.items():
    same = (df[extra] == df[base]).all()
    print(f"{extra} identical to {base}: {same}")
    assert same
df = df.drop(columns=list(redundant) + ["product_match_status", "primary_category"])

# Stable row id so later files (text lookup, model predictions) can be joined back
df = df.sort_values(["product_id", "submission_time", "author_id"], kind="mergesort").reset_index(drop=True)
df.insert(0, "review_id", np.arange(1, len(df) + 1, dtype="int64"))

cleaning_log.insert(0, {"step": "1. Load & validate", "action": "Loaded Part A merged file and ran integrity checks",
         "reason": "Confirm input quality before cleaning", "rows_before": n0, "rows_after": n0,
         "rows_removed": 0, "detail": "All integrity checks passed"})
log_step("1. Load & validate", "Dropped 3 redundant product_* copies, product_match_status, primary_category",
         "Identical to review-table columns / constant columns", len(df), len(df),
         "Added review_id as a stable unique key (kept even after later row removals)")

## Step 2. Repeated author–product pairs

Part A removed exact duplicates but found 4,660 author–product pairs with more than one review. As the proposal says, we look at **dates and text** before deciding, because a repeated pair is not automatically a duplicate:

* reviews on **different dates** → likely a repurchase and a new review → keep;
* reviews on the **same date with different text** → often different shades/variants of the same product ID (e.g. two lip-balm flavours) → keep;
* reviews on the **same date with (almost) the same text** (similarity ≥ 0.9; only the title, a profile field or feedback counts differ) → the same review submitted twice → keep the first copy only.

In [ ]:
grp = df.groupby(["author_id", "product_id"])
df["n_reviews_same_author_product"] = grp["review_id"].transform("size").astype("int16")
rep = df[df["n_reviews_same_author_product"] > 1].copy()

pair_summary = rep.groupby(["author_id", "product_id"]).agg(
    n=("review_id", "size"),
    n_dates=("submission_time", "nunique"),
    n_ratings=("rating", "nunique"),
    span_days=("submission_time", lambda s: (s.max() - s.min()).days),
)
print("Repeated pairs:", len(pair_summary), "| rows involved:", len(rep),
      f"({len(rep)/len(df):.2%} of reviews)")
print("Pairs whose reviews are all on different dates:", int((pair_summary.n_dates == pair_summary.n).sum()))
print("Median days between first and last review in a pair:", pair_summary.span_days.median())

# Same author + product + date: compare the review texts
from difflib import SequenceMatcher
same_day_key = ["author_id", "product_id", "submission_time"]
same_day = df[df.duplicated(subset=same_day_key, keep=False)].copy()
print("Rows sharing author + product + date:", len(same_day),
      "| groups:", same_day.groupby(same_day_key).ngroups)

def norm_text(t):
    return "" if pd.isna(t) else " ".join(str(t).lower().split())

TEXT_SIM_THRESHOLD = 0.9
resubmission_ids, sims = [], []
for _, grp in same_day.groupby(same_day_key):
    kept = []
    for rid, txt in zip(grp["review_id"], grp["review_text"].map(norm_text)):
        best = max((SequenceMatcher(None, k, txt).ratio() for k in kept), default=0.0)
        if kept:
            sims.append(best)
        if txt and best >= TEXT_SIM_THRESHOLD:
            resubmission_ids.append(rid)       # near-identical text: a second copy
        else:
            kept.append(txt)                   # different text: a separate review

sims = pd.Series(sims)
print("Text similarity to the closest earlier same-day review:")
print(pd.cut(sims, [0, .3, .5, .8, .9, 1.0001], include_lowest=True).value_counts().sort_index().to_string())
print("Near-identical copies (similarity >= 0.9):", len(resubmission_ids))

example = same_day[same_day.groupby(same_day_key)["review_id"].transform("size") == 2].head(6)
example.assign(review_text=example["review_text"].str[:70])[same_day_key + ["rating", "review_title", "review_text"]]

In [ ]:
# Decision: remove near-identical same-day copies (keep the first);
# keep different-text same-day reviews and different-date repeats, and flag them.
n_before = len(df)
df = df[~df["review_id"].isin(resubmission_ids)].reset_index(drop=True)
log_step("2. Repeated author-product pairs",
         "Removed same author + product + date repeats whose text is near-identical (similarity >= 0.9), kept first",
         "Same review submitted twice (only title / profile fields / feedback differ)", n_before, len(df),
         f"{n_before - len(df)} rows removed; same-day reviews with different text kept (often different shades/variants)")

df["n_reviews_same_author_product"] = (
    df.groupby(["author_id", "product_id"])["review_id"].transform("size").astype("int16"))
df["repeat_author_product"] = df["n_reviews_same_author_product"] > 1
log_step("2. Repeated author-product pairs",
         "Kept other repeats (different dates or different text); added flag repeat_author_product",
         "Likely repurchases or different variants; deleting them would lose real reviews. Flag allows a sensitivity check",
         len(df), len(df), f"{int(df['repeat_author_product'].sum())} rows flagged")
print("Rows now:", len(df), "| flagged repeat rows:", int(df["repeat_author_product"].sum()))

## Step 3. Outcome variables and class balance

* **Main outcome** `five_star` = 1 if rating = 5, else 0. No missing values.
* **Secondary outcome** `recommended` = `is_recommended` (1/0), **left missing** where the reviewer did not answer (not imputed — guessing an outcome would create fake data). Part F drops missing rows for that analysis. The year table below shows the missingness is structural: the field is empty for every review before 2017.

The course asks us to check the target distribution against a majority-class baseline: if we always predicted the majority class, accuracy would equal the majority share.

In [ ]:
df["five_star"] = (df["rating"] == 5).astype("int8")
df["recommended"] = df["is_recommended"].astype("Int8")

dist = pd.DataFrame({
    "outcome": ["five_star (main)", "recommended (secondary)"],
    "positive_share": [df["five_star"].mean(), df["recommended"].mean()],
    "missing_share": [0.0, df["recommended"].isna().mean()],
})
dist["majority_baseline_accuracy"] = np.maximum(dist["positive_share"], 1 - dist["positive_share"])
display(dist.round(4))

print(df["rating"].value_counts().sort_index())

# Is 'recommended' missing at random over time?
rec_missing_by_year = (df.assign(year=df["submission_time"].dt.year)
                         .groupby("year")["recommended"].apply(lambda s: s.isna().mean()).round(3))
print("Share of missing 'recommended' by review year:")
print(rec_missing_by_year.to_string())

log_step("3. Outcomes", "Created five_star = (rating == 5) and recommended (nullable)",
         "Main and secondary outcomes from the proposal", len(df), len(df),
         f"five_star = {df['five_star'].mean():.1%}; recommended = {df['recommended'].mean():.1%} "
         f"of non-missing, missing {df['recommended'].isna().mean():.1%} "
         f"(100% missing before 2017: the question did not exist yet, so the secondary analysis covers 2017+ only)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [1.4, 1]})
rc = df["rating"].value_counts().sort_index()
axes[0].bar(rc.index.astype(str), rc.values / 1e3,
            color=[GREY, GREY, GREY, GREY, BLUE], width=0.6)
for x, v in zip(rc.index.astype(str), rc.values):
    axes[0].text(x, v / 1e3 + 8, f"{v/len(df):.0%}", ha="center", color=INK2, fontsize=10)
axes[0].set_title("Rating distribution")
axes[0].set_ylabel("Reviews (thousands)")
axes[0].set_xlabel("Star rating")

labels = ["Five-star\n(main)", "Recommended\n(secondary)"]
pos = [df["five_star"].mean(), df["recommended"].mean()]
axes[1].barh(labels, pos, color=BLUE, height=0.5)
axes[1].barh(labels, [1 - p for p in pos], left=pos, color=GREY, height=0.5)
for i, p in enumerate(pos):
    axes[1].text(p - 0.02, i, f"{p:.0%}", ha="right", va="center", color="white", fontweight="bold")
    axes[1].text(p + 0.02, i, f"{1-p:.0%}", ha="left", va="center", color=INK)
axes[1].set_xlim(0, 1); axes[1].set_xticks([]); axes[1].grid(False)
axes[1].set_title("Class balance")
axes[1].invert_yaxis()
fig.suptitle("Five-star rating is the more balanced outcome (64/36 vs 84/16)", x=0.02, ha="left",
             fontsize=12, color=INK2)
fig.tight_layout()
fig.savefig(FIG_DIR / "B1_outcome_balance.png")
plt.show()

## Step 4. Feature selection

Each raw column is either kept as a model predictor, kept for description/sensitivity only, or excluded. The main reasons for exclusion are **data leakage** (information produced at the same time as or after the rating, or built from ratings), **time mismatch** (2023 snapshot flags applied to reviews from 2008–2023), **too many levels**, or **too much missing**.

In [ ]:
fs = [
    # column, decision, role, reason
    ("rating", "keep", "outcome source", "Defines five_star"),
    ("is_recommended", "keep (secondary outcome only)", "secondary outcome", "Given together with the rating -> leakage if used as predictor"),
    ("price_usd", "keep", "main predictor (as price_segment)", "Key product attribute; log_price for sensitivity"),
    ("secondary_category", "keep", "main predictor (as category)", "Product type; 13 levels -> 9 after merging rare ones"),
    ("highlights", "keep", "main predictor (as hl_* indicators)", "Marketing claims, e.g. fragrance-free"),
    ("skin_type", "keep", "main predictor (as skin_type_cat)", "Main reviewer variable; 7.1% missing kept as 'Not reported'"),
    ("skin_tone", "keep (descriptive only)", "descriptive", "11% missing, 14 sparse levels; used in EDA only"),
    ("submission_time", "keep", "descriptive / sensitivity", "review_year for date distribution and recent-review check"),
    ("product_id", "keep", "id / cluster", "Product-level split and clustered standard errors"),
    ("author_id", "keep", "id", "Repeat-review flag"),
    ("brand_name", "keep (not in model)", "id / descriptive", "142 brands: too many levels; target-encoding would leak"),
    ("product_name", "keep (not in model)", "id", "Labels for plots"),
    ("tertiary_category", "keep (not in model)", "descriptive", "14.7% missing and 30 levels; secondary_category is enough"),
    ("limited_edition", "keep (not in main model)", "descriptive", "2023 snapshot flag; may not hold when older reviews were written"),
    ("new", "keep (not in main model)", "descriptive", "Time-varying 2023 flag (proposal: excluded)"),
    ("online_only", "keep (not in main model)", "descriptive", "Time-varying 2023 flag (proposal: excluded)"),
    ("sephora_exclusive", "keep (not in main model)", "descriptive", "2023 snapshot flag; can change over time"),
    ("out_of_stock", "keep (not in main model)", "descriptive", "Time-varying 2023 flag"),
    ("review_text", "exclude (separate lookup file)", "leakage", "Written with the rating; saved to review_text_lookup.parquet for optional topic model"),
    ("review_title", "exclude (separate lookup file)", "leakage", "Written with the rating"),
    ("helpfulness", "exclude", "leakage", "Votes by other users after the review is posted"),
    ("total_feedback_count", "exclude", "leakage", "Post-review feedback"),
    ("total_neg_feedback_count", "exclude", "leakage", "Post-review feedback"),
    ("total_pos_feedback_count", "exclude", "leakage", "Post-review feedback"),
    ("product_rating", "exclude", "leakage", "Average of the ratings we are predicting"),
    ("reviews", "exclude", "leakage", "Product review count, built from reviews"),
    ("loves_count", "exclude", "leakage", "Popularity accumulated over time, related to ratings"),
    ("eye_color", "exclude", "not relevant", "No link to the research question; 15% missing"),
    ("hair_color", "exclude", "not relevant", "No link to the research question; 16% missing"),
    ("ingredients", "exclude", "too complex", "Free text; highlights already capture key claims"),
    ("size", "exclude", "too messy", "Free-text sizes in mixed units"),
    ("variation_type", "exclude", "not relevant", "Shade/size variant metadata"),
    ("variation_value", "exclude", "not relevant", "Shade/size variant metadata"),
    ("variation_desc", "exclude", "too much missing", "99% missing"),
    ("value_price_usd", "exclude", "too much missing", "97% missing; price_usd used instead"),
    ("sale_price_usd", "exclude", "too much missing", "99% missing; price_usd used instead"),
    ("child_count", "exclude", "not relevant", "Number of variants"),
    ("child_max_price", "exclude", "too much missing", "59% missing; overlaps price_usd"),
    ("child_min_price", "exclude", "too much missing", "59% missing; overlaps price_usd"),
    ("brand_id", "exclude", "redundant", "Same information as brand_name"),
    ("source_file", "exclude", "not relevant", "Which raw file the review came from"),
]
feature_selection = pd.DataFrame(fs, columns=["raw_column", "decision", "role", "reason"])
feature_selection["missing_share"] = feature_selection["raw_column"].map(
    lambda c: round(float(df[c].isna().mean()), 4) if c in df.columns else np.nan)
missing_cols = set(df.columns) - set(feature_selection["raw_column"]) - {
    "review_id", "n_reviews_same_author_product", "repeat_author_product", "five_star", "recommended"}
print("Raw columns not documented:", missing_cols)
assert not missing_cols
display(feature_selection.groupby("decision").size().rename("n_columns"))
n_kept = int(feature_selection["decision"].str.startswith("keep").sum())
log_step("4. Feature selection", f"Documented all {len(feature_selection)} raw columns: {n_kept} kept, "
         f"{len(feature_selection) - n_kept} excluded",
         "Exclusions: leakage, 2023 time mismatch, too many levels, >50% missing, not relevant",
         len(df), len(df), "See feature_selection.csv")
feature_selection

## Step 5. Missing values

Part A already filled skin type / tone for authors who reported the same value in their other reviews (e.g. 33,773 skin-type values), which is an evidence-based retrospective fill (see the preprocessing limitation in Step 11). What is still missing is genuinely unknown. Rule from the proposal: **do not delete rows with missing reviewer attributes**; keep "Not reported" as its own category. Reason: if missingness is not random, dropping those rows biases group comparisons. The table below shows that reviewers who do not report skin type give five stars more often (66% vs 64%), so missingness is informative.

In [ ]:
key_cols = ["rating", "price_usd", "secondary_category", "skin_type", "skin_tone",
            "highlights", "is_recommended", "tertiary_category"]
missing_tbl = pd.DataFrame({
    "missing_rows": df[key_cols].isna().sum(),
    "missing_share": df[key_cols].isna().mean().round(4),
})
missing_tbl["handling"] = missing_tbl.index.map({
    "rating": "None needed",
    "price_usd": "None needed",
    "secondary_category": "None needed",
    "skin_type": "'Not reported' category",
    "skin_tone": "'Not reported' category (descriptive only)",
    "highlights": "Indicators = 0 + highlights_missing flag",
    "is_recommended": "Left missing (field absent before 2017; not imputed)",
    "tertiary_category": "Not used",
})
display(missing_tbl)

five_by_missing = df.groupby(df["skin_type"].isna().map({True: "Not reported", False: "Reported"}))["five_star"].agg(["mean", "size"])
print("Five-star rate by whether skin type is reported:")
display(five_by_missing.round(4))

In [ ]:
plot_tbl = missing_tbl.drop(index=["rating", "price_usd", "secondary_category"]).sort_values("missing_share")
nice = {"is_recommended": "Recommended flag", "tertiary_category": "Tertiary category",
        "skin_tone": "Skin tone", "highlights": "Highlights", "skin_type": "Skin type"}
fig, ax = plt.subplots(figsize=(7.5, 3.2))
ax.barh([nice[i] for i in plot_tbl.index], plot_tbl["missing_share"] * 100, color=BLUE, height=0.55)
for i, (share, how) in enumerate(zip(plot_tbl["missing_share"], plot_tbl["handling"])):
    ax.text(share * 100 + 0.4, i, f"{share:.1%}  ->  {how}", va="center", color=INK2, fontsize=9.5)
ax.set_xlim(0, 40)
ax.set_xlabel("% of reviews missing")
ax.set_title("Missing values are kept and labelled, not deleted", loc="left")
ax.grid(axis="y", visible=False)
fig.savefig(FIG_DIR / "B2_missing_values.png")
plt.show()

log_step("5. Missing values", "skin_type / skin_tone missing -> 'Not reported'; highlights missing -> flag",
         "Missingness is informative (five-star rate 66% vs 64%); deleting would bias comparisons",
         len(df), len(df), "No rows removed")

## Step 6. Rare categories — merge by meaning, do not delete

**Product category (`secondary_category` → `category`).** Rule: a category with < 2% of reviews is merged into the closest category by meaning. Five categories fall under 2%:

* *Value & Gift Sets* (1.1%) and *Shop by Concern* (1 product, a mini mask) → join *Mini Size* as **Minis & Sets** (they are defined by format, not function).
* *Self Tanners*, *Wellness* (supplements, rollers) and *High Tech Tools* (devices) → **Tools, Wellness & Self-Tan** (non-core skincare).
* *Lip Balms & Treatments* is renamed **Lip Care**.

**Skin type.** Only 4 reported levels, all large → capitalised, missing → *Not reported*.

**Skin tone (descriptive only).** 14 levels, several with < 2,000 reviews (porcelain, olive, dark, ebony). Merged along Sephora's light-to-deep scale: Fair (porcelain, fair, fairLight), Light, Light-Medium, Medium (medium, mediumTan), Tan/Olive (tan, olive), Deep (deep, rich, dark, ebony), Not reported (missing, notSureST).

In [ ]:
CATEGORY_MAP = {
    "Moisturizers": "Moisturizers",
    "Treatments": "Treatments",
    "Cleansers": "Cleansers",
    "Eye Care": "Eye Care",
    "Masks": "Masks",
    "Lip Balms & Treatments": "Lip Care",
    "Sunscreen": "Sunscreen",
    "Mini Size": "Minis & Sets",
    "Value & Gift Sets": "Minis & Sets",
    "Shop by Concern": "Minis & Sets",
    "Self Tanners": "Tools, Wellness & Self-Tan",
    "Wellness": "Tools, Wellness & Self-Tan",
    "High Tech Tools": "Tools, Wellness & Self-Tan",
}
CATEGORY_LEVELS = ["Moisturizers", "Treatments", "Cleansers", "Eye Care", "Masks",
                   "Lip Care", "Sunscreen", "Minis & Sets", "Tools, Wellness & Self-Tan"]
assert set(df["secondary_category"].unique()) <= set(CATEGORY_MAP)

before = (df.groupby("secondary_category")
            .agg(products=("product_id", "nunique"), reviews=("review_id", "size"),
                 five_star_rate=("five_star", "mean")))
before["review_share"] = before["reviews"] / len(df)
before["merged_into"] = before.index.map(CATEGORY_MAP)
before["rare (<2%)"] = before["review_share"] < 0.02
display(before.sort_values("reviews", ascending=False).round(4))

df["category"] = pd.Categorical(df["secondary_category"].map(CATEGORY_MAP), categories=CATEGORY_LEVELS)
after = (df.groupby("category", observed=True)
           .agg(products=("product_id", "nunique"), reviews=("review_id", "size"),
                five_star_rate=("five_star", "mean")))
after["review_share"] = after["reviews"] / len(df)
display(after.round(4))

log_step("6. Rare categories", "secondary_category 13 levels -> category 9 levels",
         "Categories with <2% of reviews merged by meaning (format-based -> Minis & Sets; "
         "non-core -> Tools, Wellness & Self-Tan)", len(df), len(df),
         "Value & Gift Sets, Shop by Concern -> Minis & Sets; Self Tanners, Wellness, High Tech Tools -> Tools, Wellness & Self-Tan")

In [ ]:
SKIN_TYPE_MAP = {"normal": "Normal", "combination": "Combination", "dry": "Dry", "oily": "Oily"}
SKIN_TYPE_LEVELS = ["Normal", "Combination", "Dry", "Oily", "Not reported"]
assert set(df["skin_type"].dropna().unique()) <= set(SKIN_TYPE_MAP)
df["skin_type_cat"] = pd.Categorical(df["skin_type"].map(SKIN_TYPE_MAP).fillna("Not reported"),
                                     categories=SKIN_TYPE_LEVELS)

SKIN_TONE_MAP = {
    "porcelain": "Fair", "fair": "Fair", "fairLight": "Fair",
    "light": "Light",
    "lightMedium": "Light-Medium",
    "medium": "Medium", "mediumTan": "Medium",
    "tan": "Tan/Olive", "olive": "Tan/Olive",
    "deep": "Deep", "rich": "Deep", "dark": "Deep", "ebony": "Deep",
    "notSureST": "Not reported",
}
SKIN_TONE_LEVELS = ["Fair", "Light", "Light-Medium", "Medium", "Tan/Olive", "Deep", "Not reported"]
assert set(df["skin_tone"].dropna().unique()) <= set(SKIN_TONE_MAP)
df["skin_tone_group"] = pd.Categorical(df["skin_tone"].map(SKIN_TONE_MAP).fillna("Not reported"),
                                       categories=SKIN_TONE_LEVELS)

display(df["skin_type_cat"].value_counts(sort=False).to_frame("reviews"))
display(pd.crosstab(df["skin_tone"].fillna("<missing>"), df["skin_tone_group"]).loc[
    lambda t: t.sum(axis=1).sort_values(ascending=False).index])

log_step("6. Rare categories", "skin_type -> skin_type_cat (5 levels incl. 'Not reported')",
         "Standardise labels; keep missing as a category", len(df), len(df), "")
log_step("6. Rare categories", "skin_tone 14 levels -> skin_tone_group 7 levels",
         "Sparse tones (porcelain 1.7k, olive 1.8k, dark 539, ebony 3) merged along the light-deep scale",
         len(df), len(df), "Descriptive use only")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
b = before.sort_values("reviews")
colors = [GREY if r else BLUE for r in b["rare (<2%)"]]
ax.barh(b.index, b["review_share"] * 100, color=colors, height=0.6)
for i, (name, row) in enumerate(b.iterrows()):
    label = f"{row.review_share:.1%}"
    if row["rare (<2%)"]:
        label += f"  ->  {row.merged_into}"
    ax.text(max(row.review_share * 100, 2) + 0.5, i, label, va="center", fontsize=9.5,
            color=INK2 if row["rare (<2%)"] else INK)
ax.axvline(2, color=ORANGE, lw=1.5, ls="--")
ax.text(2.4, len(b) - 0.4, "2% threshold", color=ORANGE, fontsize=9.5, va="bottom")
ax.set_ylim(-0.6, len(b) + 0.4)
ax.set_xlabel("% of reviews")
ax.set_xlim(0, 36)
ax.grid(axis="y", visible=False)
ax.set_title("13 product categories -> 9: rare ones merged by meaning", loc="left")
fig.savefig(FIG_DIR / "B3_category_merge.png")
plt.show()

## Step 7. Price segments and log price

Price is right-skewed (median product $44, max $1,900), so we do not use raw price. We use **fixed, round-number cut-offs** rather than quantiles: they are easy to explain on a slide and do not depend on the data, so there is no risk that a train/test split changes the definition.

| Segment | Price | Approx. position among products |
|---|---|---|
| Budget (ref.) | < $25 | lowest ~20% |
| Mid | $25–49 | ~20th–60th percentile |
| Premium | $50–99 | ~60th–90th percentile |
| Luxury | ≥ $100 | top ~10% |

`log_price` (natural log of `price_usd`) is created for Part F's sensitivity check only, so the main model does not contain two overlapping price variables.

In [ ]:
PRICE_BINS = [0, 25, 50, 100, np.inf]
PRICE_LEVELS = ["Budget (<$25)", "Mid ($25-49)", "Premium ($50-99)", "Luxury ($100+)"]
df["price_segment"] = pd.cut(df["price_usd"].astype(float), bins=PRICE_BINS, labels=PRICE_LEVELS, right=False)
df["log_price"] = np.log(df["price_usd"].astype(float))
assert df["price_segment"].notna().all()

prod = df.drop_duplicates("product_id")
print("Product-level price percentiles:", prod["price_usd"].astype(float).quantile([.2, .5, .6, .9, .99]).round(1).to_dict())
seg_tbl = pd.DataFrame({
    "products": prod["price_segment"].value_counts(sort=False),
    "reviews": df["price_segment"].value_counts(sort=False),
    "five_star_rate": df.groupby("price_segment", observed=True)["five_star"].mean().round(4),
})
seg_tbl["product_share"] = (seg_tbl["products"] / seg_tbl["products"].sum()).round(3)
display(seg_tbl)

print("Products priced above $500 (checked, real luxury items, kept):")
display(prod.loc[prod["price_usd"] > 500, ["product_id", "brand_name", "product_name", "price_usd"]])

log_step("7. Price", "Created price_segment (4 fixed bins) and log_price",
         "Skewed price; fixed cut-offs are interpretable and split-independent", len(df), len(df),
         "Bins: <25, 25-49, 50-99, 100+ USD; outliers >$500 kept (genuine luxury products)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
logp = np.log10(prod["price_usd"].astype(float))
ax.hist(logp, bins=45, color=BLUE, edgecolor="white", linewidth=0.6)
for cut in [25, 50, 100]:
    ax.axvline(np.log10(cut), color=ORANGE, lw=1.5, ls="--")
ticks = [5, 10, 25, 50, 100, 250, 500, 1000]
ax.set_xticks(np.log10(ticks)); ax.set_xticklabels([f"${t}" for t in ticks])
ymax = ax.get_ylim()[1] * 1.3
ax.set_ylim(0, ymax)
mids = [np.log10(11), np.log10(35.4), np.log10(70.7), np.log10(220)]
for m, lab, n in zip(mids, PRICE_LEVELS, seg_tbl["products"]):
    ax.text(m, ymax * 0.97, f"{lab.split(' (')[0]}\n{n}", ha="center", va="top", fontsize=9.5, color=INK)
ax.text(np.log10(3), ymax * 0.97, "Products\nper segment:", ha="left", va="top", fontsize=9, color=INK2)
ax.set_xlabel("Product price (log scale)")
ax.set_ylabel("Products")
ax.set_title("Price is right-skewed: four fixed price segments", loc="left")
fig.savefig(FIG_DIR / "B4_price_segments.png")
plt.show()

## Step 8. Highlights indicators

`highlights` is a list stored as text, e.g. `"['Vegan', 'Fragrance Free', 'Hydrating']"`. We parse it and count tags at the **product** level (85 distinct tags). Testing all 85 would be fishing for significant results, so we pre-specify 8 indicators with a transparent rule. Tags are grouped into families and we take the most common tags of each family, **using tag frequency only (never the outcome)**:

| Family | Example tags | Selected | Why |
|---|---|---|---|
| Values badges | Clean at Sephora, Vegan, Cruelty-Free, Clean + Planet Positive | top 3: **Clean, Vegan, Cruelty-Free** | *Clean + Planet Positive* is an add-on to *Clean at Sephora* |
| Skin concern ("Good for: …") | Dullness, Dryness, Anti-Aging, Loss of firmness, Pores | top 3: **Dullness, Dryness, Anti-Aging** | the most common concerns |
| Benefit claims | Hydrating, Plumping | top 1: **Hydrating** | |
| Free-from claims | Without Parabens, Fragrance Free, Without Sulfates | **Fragrance Free** | named in the proposal; the free-from claim most relevant to skin reaction. *Without Parabens/Sulfates* are covered by the Clean standard |
| Ingredients | Hyaluronic Acid, Vitamin C, Niacinamide | none | overlap with benefit and concern claims |
| "Best for … skin" | Best for Dry, Combo, Normal Skin | used in `skin_type_match` | |
| Popularity / awards | Community Favorite, allure Best of Beauty | **excluded** | *Community Favorite* is awarded for customer love and reviews (**leakage**); awards are year-specific and partly popularity-based |

Products with no highlights (414 products, 10% of reviews) get 0 on every indicator **plus** `highlights_missing = 1`, so "no information" is not confused with "no claim".

We also build `skin_type_match` for exploration: whether the product says it is *Best for* the reviewer's own skin type (only defined when the product has such a tag and the reviewer reported a skin type).

In [ ]:
def parse_tags(s):
    if s is None or pd.isna(s):
        return None
    try:
        tags = ast.literal_eval(s)
    except (ValueError, SyntaxError) as exc:
        raise ValueError(f"Malformed highlights: {s!r}") from exc
    if not isinstance(tags, (list, tuple)) or not all(isinstance(t, str) for t in tags):
        raise ValueError(f"Highlights must contain a list of strings: {s!r}")
    return [t.strip() for t in tags]

prod_tags = (df.drop_duplicates("product_id").set_index("product_id")["highlights"].map(parse_tags))
print("Products with missing highlights:", int(prod_tags.isna().sum()), "of", len(prod_tags))

tag_counts = (prod_tags.dropna().explode().value_counts().rename("products").to_frame())
tag_counts["product_share"] = (tag_counts["products"] / prod_tags.notna().sum()).round(3)
print("Distinct tags:", len(tag_counts))
display(tag_counts.head(25))

In [ ]:
HIGHLIGHT_FEATURES = {
    "hl_fragrance_free": "Fragrance Free",
    "hl_clean": "Clean at Sephora",
    "hl_vegan": "Vegan",
    "hl_cruelty_free": "Cruelty-Free",
    "hl_hydrating": "Hydrating",
    "hl_dullness": "Good for: Dullness/Uneven Texture",
    "hl_dryness": "Good for: Dryness",
    "hl_anti_aging": "Good for: Anti-Aging",
}
EXCLUDED_TAGS = ["Community Favorite"] + [t for t in tag_counts.index if t.lower().startswith("allure")]
print("Excluded (leakage / time mismatch):", EXCLUDED_TAGS)

# Show the selection rule in the data: top tags per family (frequency only)
def tag_family(t):
    if t in ["Clean at Sephora", "Vegan", "Cruelty-Free", "Clean + Planet Positive", "Black Owned at Sephora"]:
        return "Values badges"
    if t.startswith("Good for:"):
        return "Skin concern"
    if t in ["Hydrating", "Plumping"]:
        return "Benefit claims"
    if t.startswith("Without") or t.endswith("Free") and t not in ["Cruelty-Free"]:
        return "Free-from claims"
    if t.startswith("Best for"):
        return "Best for skin type"
    if t in EXCLUDED_TAGS:
        return "Popularity / awards"
    return "Other (ingredients, finish, packaging, ...)"
fam = tag_counts.assign(family=tag_counts.index.map(tag_family),
                        selected=tag_counts.index.isin(HIGHLIGHT_FEATURES.values()))
display(fam.sort_values(["family", "products"], ascending=[True, False]).groupby("family").head(4))
for tag in HIGHLIGHT_FEATURES.values():
    assert tag in tag_counts.index, tag

prod_feat = pd.DataFrame(index=prod_tags.index)
prod_feat["highlights_missing"] = prod_tags.isna()
for col, tag in HIGHLIGHT_FEATURES.items():
    prod_feat[col] = prod_tags.map(lambda ts: ts is not None and tag in ts)
# Count of tags, leaving out the leakage / award tags
prod_feat["n_highlights"] = prod_tags.map(
    lambda ts: 0 if ts is None else sum(t not in EXCLUDED_TAGS for t in ts)).astype("int16")

# Skin-type target from "Best for ..." tags
BEST_FOR = {
    "Best for Dry, Combo, Normal Skin": {"Dry", "Combination", "Normal"},
    "Best for Oily, Combo, Normal Skin": {"Oily", "Combination", "Normal"},
    "Best for Dry Skin": {"Dry"},
    "Best for Oily Skin": {"Oily"},
    "Best for Normal Skin": {"Normal"},
    "Best for Combination Skin": {"Combination"},
}
def target_types(ts):
    if ts is None:
        return None
    s = set()
    for t in ts:
        s |= BEST_FOR.get(t, set())
    return frozenset(s) if s else None
prod_feat["_target_types"] = prod_tags.map(target_types)

shares = prod_feat[list(HIGHLIGHT_FEATURES)].mean().round(3).rename("product_share")
display(shares.to_frame())

# Correlation check between indicators (product level, products with highlights)
corr = prod_feat.loc[~prod_feat["highlights_missing"], list(HIGHLIGHT_FEATURES)].astype(int).corr().round(2)
display(corr)
max_off_diag = corr.where(~np.eye(len(corr), dtype=bool)).abs().max().max()
print("Largest correlation between two indicators:", max_off_diag)
assert max_off_diag < 0.7, "Two indicators are nearly redundant; drop one"

In [ ]:
df = df.merge(prod_feat, left_on="product_id", right_index=True, how="left", validate="many_to_one")
for col in ["highlights_missing"] + list(HIGHLIGHT_FEATURES):
    df[col] = df[col].astype("int8")

has_target = df["_target_types"].notna() & (df["skin_type_cat"] != "Not reported")
df["skin_type_match"] = pd.array([pd.NA] * len(df), dtype="Int8")
df.loc[has_target, "skin_type_match"] = [
    int(st in tt) for st, tt in zip(df.loc[has_target, "skin_type_cat"], df.loc[has_target, "_target_types"])]
df = df.drop(columns=["_target_types"])
print("skin_type_match defined for", f"{has_target.mean():.1%}", "of reviews")
display(df.groupby("skin_type_match")["five_star"].agg(["mean", "size"]).round(4))

hl_rates = pd.DataFrame({
    "with_claim": [df.loc[df[c] == 1, "five_star"].mean() for c in HIGHLIGHT_FEATURES],
    "without_claim": [df.loc[(df[c] == 0) & (df["highlights_missing"] == 0), "five_star"].mean() for c in HIGHLIGHT_FEATURES],
    "reviews_with_claim": [int(df[c].sum()) for c in HIGHLIGHT_FEATURES],
}, index=list(HIGHLIGHT_FEATURES)).round(4)
print("Raw five-star rate with vs without each claim (no adjustment, for orientation only):")
display(hl_rates)

log_step("8. Highlights", "Parsed highlights; created 8 hl_* indicators, highlights_missing, n_highlights",
         "Pre-specified by tag family using tag frequency only (never the outcome); max pairwise correlation "
         f"{max_off_diag:.2f}", len(df), len(df),
         "Excluded 'Community Favorite' and allure awards (leakage / time mismatch)")
log_step("8. Highlights", "Created skin_type_match (exploratory)",
         "Product's 'Best for' skin type vs reviewer's skin type", len(df), len(df),
         f"Defined for {has_target.mean():.1%} of reviews; NA otherwise")

## Step 9. Time variables

Product information was scraped in March 2023 while reviews go back to 2008, so older reviews may be matched with a price or claim that did not apply at the time. We add `review_year` (for the date distribution in EDA) and `recent_review` (submitted 2020 or later) so Part F can re-run the model on recent reviews as a sensitivity check.

In [ ]:
df["review_date"] = df["submission_time"].dt.normalize()
df["review_year"] = df["submission_time"].dt.year.astype("int16")
df["recent_review"] = (df["review_year"] >= 2020).astype("int8")
print("Share of reviews from 2020 onwards:", round(df["recent_review"].mean(), 3))
display(df.groupby("review_year").agg(reviews=("review_id", "size"), five_star_rate=("five_star", "mean")).round(3).T)
log_step("9. Time", "Created review_date, review_year, recent_review (>=2020)",
         "Supports date EDA and a recent-reviews sensitivity check", len(df), len(df),
         f"{df['recent_review'].mean():.1%} of reviews are from 2020+")

## Step 10. Categorical encoding

Two forms are saved so every teammate can use their preferred tool:

1. **pandas `category` columns with a fixed level order** — the first level is the reference. Works directly with `statsmodels` formulas, e.g. `C(price_segment)`.
2. **One-hot dummy columns `x_*` with the reference level dropped** — ready for `scikit-learn`.

Reference levels: **Budget (<$25)** for price, **Moisturizers** (largest category) for category, **Normal** for skin type. Because the levels are fixed in code (not learned from the data), encoding does not leak test information; Part E still has to split by `product_id`.

In [ ]:
REFERENCE = {"price_segment": PRICE_LEVELS[0], "category": CATEGORY_LEVELS[0], "skin_type_cat": SKIN_TYPE_LEVELS[0]}
df["price_segment"] = df["price_segment"].cat.set_categories(PRICE_LEVELS)

def slug(s):
    return (s.lower().replace("(", "").replace(")", "").replace("$", "").replace("+", "plus")
             .replace("<", "lt").replace(",", "").replace("&", "and").replace("/", "_")
             .replace("-", "_").replace(" ", "_").replace("__", "_").strip("_"))

PREFIX = {"price_segment": "x_price_", "category": "x_cat_", "skin_type_cat": "x_skin_"}
dummy_cols = []
for col, prefix in PREFIX.items():
    levels = list(df[col].cat.categories)
    for lev in levels[1:]:  # drop reference
        name = prefix + slug(lev.split(" (")[0] if col == "price_segment" else lev)
        df[name] = (df[col] == lev).astype("int8")
        dummy_cols.append(name)
print("Dummy columns:", dummy_cols)

# Check: each row has at most one dummy per variable, and reference rows have none
for col, prefix in PREFIX.items():
    cols = [c for c in dummy_cols if c.startswith(prefix)]
    s = df[cols].sum(axis=1)
    assert s.max() == 1
    assert ((s == 0) == (df[col] == REFERENCE[col])).all()
print("Encoding checks passed. Reference levels:", REFERENCE)

log_step("10. Encoding", "Categorical columns with fixed level order + drop-first dummies (x_*)",
         "statsmodels formulas and scikit-learn both supported; levels fixed in code", len(df), len(df),
         f"{len(dummy_cols)} dummy columns; references: {REFERENCE}")

## Step 11. Data-leakage check

Leakage = using information that would not be available when predicting the rating, or that is built from the rating itself. The check below (a) asserts that no leakage column is in the main predictor list, and (b) shows *why* these columns are dangerous: they are almost the answer.

In [ ]:
MAIN_PREDICTORS = ["price_segment", "category", "skin_type_cat", "highlights_missing"] + list(HIGHLIGHT_FEATURES)
MAIN_PREDICTORS_DUMMIES = dummy_cols + ["highlights_missing"] + list(HIGHLIGHT_FEATURES)
LEAKAGE = ["rating", "is_recommended", "recommended", "review_text", "review_title", "helpfulness",
           "total_feedback_count", "total_neg_feedback_count", "total_pos_feedback_count",
           "product_rating", "reviews", "loves_count"]
TIME_VARYING_2023 = ["new", "online_only", "limited_edition", "out_of_stock", "sephora_exclusive"]

assert not set(MAIN_PREDICTORS) & set(LEAKAGE + TIME_VARYING_2023)
assert not set(MAIN_PREDICTORS_DUMMIES) & set(LEAKAGE + TIME_VARYING_2023)
assert "Community Favorite" not in HIGHLIGHT_FEATURES.values()
print("No leakage or time-varying column among main predictors.")

# Why it matters
p5_rec = df.groupby("recommended")["five_star"].mean()
prod_lvl = df.groupby("product_id").agg(product_rating=("product_rating", "first"),
                                        five_star_rate=("five_star", "mean"), n=("review_id", "size"))
leak_demo = pd.DataFrame({
    "evidence": ["P(five-star | recommended = 1)", "P(five-star | recommended = 0)",
                 "Correlation: product average rating vs product five-star rate (products with >= 20 reviews)"],
    "value": [p5_rec.get(1), p5_rec.get(0),
              prod_lvl.query("n >= 20")[["product_rating", "five_star_rate"]].astype(float).corr().iloc[0, 1]],
}).round(3)
display(leak_demo)

log_step("11. Leakage", "Asserted that main predictors exclude leakage and 2023 time-varying columns",
         "Recommendation, text, feedback counts and product rating summaries are produced with / from ratings",
         len(df), len(df),
         f"P(5*|rec=1)={p5_rec.get(1):.2f} vs P(5*|rec=0)={p5_rec.get(0):.2f}")

**Leakage through preprocessing itself.** A second kind of leakage happens when a cleaning step *learns* something from the whole data set (including the future test set) or from the outcome. The saved transformations are fixed for downstream reruns. However, category merging and highlight selection were informed by the full predictor distribution, and Part A filled author attributes across products before the split. Thus this is a retrospective evaluation using globally prepared features, not a strictly isolated prospective preprocessing evaluation. A future prospective analysis should define these choices on training products and restrict author fills to information available at prediction time:

| Step | Learned from data? | Uses the outcome? |
|---|---|---|
| Price segments | No: fixed cut-offs $25 / $50 / $100 | No |
| log_price | No: row-wise | No |
| Category merging | Rule uses review share of each category (predictor distribution) | No |
| Skin type / tone labels | No: fixed mapping | No |
| Highlight selection | Tag frequency only | No |
| Encoding | No: levels fixed in code | No |
| Missing values | Part A uses the same author's other reviews, possibly across train/test products | No |

## Step 12. Model-readiness checks

Checks that protect Parts D and E:

1. **Product-level variables are constant within a product.** This is required for a product-level train/test split and for clustered standard errors to make sense.
2. **Every skin type × price segment cell is well populated.** The pre-specified interaction needs enough reviews *and* enough products per cell, because the effective sample size is closer to the number of products than to the number of reviews.
3. **Enough products per category** for a product-level split to put every category in both train and test.

In [ ]:
PRODUCT_LEVEL = ["price_usd", "price_segment", "log_price", "category", "highlights_missing", "n_highlights"] + list(HIGHLIGHT_FEATURES)
n_varying = df.groupby("product_id")[PRODUCT_LEVEL].nunique(dropna=False).gt(1).sum()
print("Products where a product-level variable varies across reviews:")
print(n_varying.to_string())
assert n_varying.sum() == 0

cells = (df.groupby(["skin_type_cat", "price_segment"], observed=True)
           .agg(reviews=("review_id", "size"), products=("product_id", "nunique"), five_star_rate=("five_star", "mean")))
print("\nReviews per skin type x price segment:")
display(cells["reviews"].unstack())
print("Products per cell:")
display(cells["products"].unstack())
print("Raw five-star rate per cell (no adjustment):")
display(cells["five_star_rate"].unstack().round(3))
assert cells["reviews"].min() >= 1000 and cells["products"].min() >= 100

prod_per_cat = df.groupby("category", observed=True)["product_id"].nunique()
print("Fewest products in a category:", prod_per_cat.idxmin(), int(prod_per_cat.min()))
assert prod_per_cat.min() >= 50

n_params = 1 + (len(PRICE_LEVELS) - 1) * len(SKIN_TYPE_LEVELS) + (len(SKIN_TYPE_LEVELS) - 1) \
           + (len(CATEGORY_LEVELS) - 1) + 1 + len(HIGHLIGHT_FEATURES)
minority = int(min(df["five_star"].sum(), (1 - df["five_star"]).sum()))
print(f"Main model parameters: {n_params}; minority-class events: {minority:,} "
      f"({minority / n_params:,.0f} events per parameter)")

log_step("12. Model readiness", "Checked product-level constancy, interaction cell sizes, products per category",
         "Product-level split and clustered SEs need constant product variables; interaction needs populated cells",
         len(df), len(df),
         f"Smallest cell: {int(cells['reviews'].min()):,} reviews / {int(cells['products'].min())} products; "
         f"{n_params} parameters")

## Step 13. Assemble, save and validate

In [ ]:
ID_COLS = ["review_id", "product_id", "author_id", "brand_name", "product_name"]
OUTCOME_COLS = ["rating", "five_star", "recommended"]
PREDICTOR_COLS = ["price_usd", "price_segment", "log_price", "category", "skin_type_cat",
                  "highlights_missing"] + list(HIGHLIGHT_FEATURES)
EXTRA_COLS = ["n_highlights", "skin_type_match", "skin_tone_group", "review_date", "review_year",
              "recent_review", "repeat_author_product", "n_reviews_same_author_product",
              "secondary_category", "tertiary_category"] + TIME_VARYING_2023
final_cols = ID_COLS + OUTCOME_COLS + PREDICTOR_COLS + dummy_cols + EXTRA_COLS

out = df[final_cols].copy()
out["price_usd"] = out["price_usd"].astype("float64")
out["rating"] = out["rating"].astype("int8")
for c in TIME_VARYING_2023 + ["repeat_author_product"]:
    out[c] = out[c].astype(bool).astype("int8")
for c in ["product_id", "author_id", "brand_name", "product_name", "secondary_category", "tertiary_category"]:
    out[c] = out[c].astype("string")

# Upstream rebuilds write to the external/ignored data directory, preserving the supplied file.
ANALYSIS_PATH = PROCESSED_DIR / "analysis_ready.parquet"
TEXT_PATH = PROCESSED_DIR / "review_text_lookup.parquet"
if ANALYSIS_PATH.exists() or TEXT_PATH.exists():
    raise FileExistsError("Processed outputs already exist; choose a new SEPHORA_DATA_DIR.")
out.to_parquet(ANALYSIS_PATH, index=False, engine="pyarrow")
df[["review_id", "product_id", "rating", "review_title", "review_text"]].to_parquet(TEXT_PATH, index=False, engine="pyarrow")

log_step("13. Save", "Saved analysis_ready.parquet and review_text_lookup.parquet",
         "Final analysis file for Parts C-F", n0, len(out),
         f"{out.shape[1]} columns; text stored separately")

# Validation: read back
chk = pd.read_parquet(ANALYSIS_PATH)
assert chk.shape == out.shape
assert list(chk["price_segment"].cat.categories) == PRICE_LEVELS
assert list(chk["category"].cat.categories) == CATEGORY_LEVELS
assert list(chk["skin_type_cat"].cat.categories) == SKIN_TYPE_LEVELS
assert chk["review_id"].is_unique
assert chk[PREDICTOR_COLS + dummy_cols + ["five_star"]].isna().sum().sum() == 0
print("Saved:", ANALYSIS_PATH, chk.shape, f"{ANALYSIS_PATH.stat().st_size/1e6:.1f} MB")
print("Saved:", TEXT_PATH, f"{TEXT_PATH.stat().st_size/1e6:.1f} MB")
print("Validation passed: categories, unique ids, no missing in outcome / predictors.")

In [ ]:
def describe_levels(s):
    if isinstance(s.dtype, pd.CategoricalDtype):
        return " | ".join(f"{c}{' (ref)' if i == 0 else ''}" for i, c in enumerate(s.cat.categories))
    if s.dropna().isin([0, 1]).all() and s.nunique() <= 2:
        return "0 / 1"
    if pd.api.types.is_numeric_dtype(s):
        return f"min {s.min():.4g}, median {s.median():.4g}, max {s.max():.4g}"
    if pd.api.types.is_datetime64_any_dtype(s):
        return f"{s.min().date()} to {s.max().date()}"
    return f"{s.nunique()} unique values"

DICT_INFO = {
    "review_id": ("Unique review key (assigned before de-duplication, so numbers have gaps)", "created", "id"),
    "product_id": ("Sephora product ID; use for product-level train/test split and clustered SEs", "reviews", "id / cluster"),
    "author_id": ("Reviewer ID", "reviews", "id"),
    "brand_name": ("Brand", "reviews", "id / descriptive"),
    "product_name": ("Product name", "reviews", "id / descriptive"),
    "rating": ("Star rating 1-5 (do NOT use as predictor)", "reviews", "outcome source"),
    "five_star": ("1 if rating = 5, else 0", "derived from rating", "MAIN OUTCOME"),
    "recommended": ("1 = recommends, 0 = does not, NA = not answered (do NOT use as predictor)", "reviews.is_recommended", "SECONDARY OUTCOME"),
    "price_usd": ("Product list price in USD (2023 snapshot)", "products", "raw predictor"),
    "price_segment": ("Price band: <$25, $25-49, $50-99, $100+", "derived from price_usd", "main predictor"),
    "log_price": ("Natural log of price_usd", "derived from price_usd", "sensitivity predictor (Part F)"),
    "category": ("Product category, rare levels merged by meaning (13 -> 9)", "products.secondary_category", "main predictor"),
    "skin_type_cat": ("Reviewer skin type; missing kept as 'Not reported'", "reviews.skin_type", "main predictor (interaction with price_segment)"),
    "highlights_missing": ("1 if the product has no highlights listed", "products.highlights", "main predictor (control)"),
    "hl_fragrance_free": ("Product claims 'Fragrance Free'", "products.highlights", "main predictor"),
    "hl_clean": ("Product has 'Clean at Sephora' badge", "products.highlights", "main predictor"),
    "hl_vegan": ("Product claims 'Vegan'", "products.highlights", "main predictor"),
    "hl_cruelty_free": ("Product claims 'Cruelty-Free'", "products.highlights", "main predictor"),
    "hl_hydrating": ("Product claims 'Hydrating'", "products.highlights", "main predictor"),
    "hl_anti_aging": ("Product tagged 'Good for: Anti-Aging'", "products.highlights", "main predictor"),
    "hl_dryness": ("Product tagged 'Good for: Dryness'", "products.highlights", "main predictor"),
    "hl_dullness": ("Product tagged 'Good for: Dullness/Uneven Texture'", "products.highlights", "main predictor"),
    "n_highlights": ("Number of highlight tags on the product (excluding Community Favorite and awards)", "products.highlights", "descriptive"),
    "skin_type_match": ("1 if product is 'Best for' the reviewer's skin type, 0 if not, NA if undefined", "highlights + skin_type", "exploratory"),
    "skin_tone_group": ("Reviewer skin tone, 14 -> 7 groups", "reviews.skin_tone", "descriptive only"),
    "review_date": ("Review submission date", "reviews.submission_time", "descriptive"),
    "review_year": ("Year of review", "reviews.submission_time", "descriptive"),
    "recent_review": ("1 if review submitted 2020 or later", "reviews.submission_time", "sensitivity subset"),
    "repeat_author_product": ("1 if this author reviewed this product more than once (different dates)", "created", "sensitivity subset"),
    "n_reviews_same_author_product": ("Number of reviews by this author for this product", "created", "descriptive"),
    "secondary_category": ("Original category before merging", "products", "reference"),
    "tertiary_category": ("Finer category (14.7% missing)", "products", "descriptive"),
    "new": ("2023 snapshot flag: new product", "products", "excluded from main model (time-varying)"),
    "online_only": ("2023 snapshot flag: online only", "products", "excluded from main model (time-varying)"),
    "limited_edition": ("2023 snapshot flag: limited edition", "products", "excluded from main model (time-varying)"),
    "out_of_stock": ("2023 snapshot flag: out of stock", "products", "excluded from main model (time-varying)"),
    "sephora_exclusive": ("2023 snapshot flag: Sephora exclusive", "products", "excluded from main model (time-varying)"),
}
for d in dummy_cols:
    base = next(k for k, p in PREFIX.items() if d.startswith(p))
    DICT_INFO[d] = (f"Dummy for one level of {base} (reference: {REFERENCE[base]})", f"encoded from {base}", "encoded predictor (scikit-learn)")

feature_dictionary = pd.DataFrame([
    {"column": c, "dtype": str(out[c].dtype), "description": DICT_INFO[c][0], "source": DICT_INFO[c][1],
     "role": DICT_INFO[c][2], "levels_or_range": describe_levels(out[c]),
     "missing_share": round(float(out[c].isna().mean()), 4)}
    for c in out.columns])
assert len(feature_dictionary) == out.shape[1]

cleaning_log_df = pd.DataFrame(cleaning_log)
cleaning_log_df.to_csv(OUT_DIR / "cleaning_summary.csv", index=False, encoding="utf-8-sig")
feature_dictionary.to_csv(OUT_DIR / "feature_dictionary.csv", index=False, encoding="utf-8-sig")
feature_selection.to_csv(OUT_DIR / "feature_selection.csv", index=False, encoding="utf-8-sig")
display(cleaning_log_df)

In [ ]:
display(feature_dictionary)

## Notes for Parts C–F

```python
import pandas as pd
from project_paths import require_analysis
df = pd.read_parquet(require_analysis())   # categories and reference levels are preserved
```

* **Outcome:** `five_star` (main), `recommended` (secondary; drop NA rows only for that analysis). Note: `recommended` is missing for **every** review before 2017 (the field did not exist yet) and almost never after 2018, so the secondary analysis describes 2017+ reviews only. Compare it with the main model fitted on the same years.
* **Main model (statsmodels):**
  `five_star ~ C(price_segment) * C(skin_type_cat) + C(category) + highlights_missing + hl_fragrance_free + hl_clean + hl_vegan + hl_cruelty_free + hl_hydrating + hl_dullness + hl_dryness + hl_anti_aging`
  with standard errors clustered by `product_id`. The first level of each category is the reference.
* **scikit-learn:** use the `x_*` dummies + `highlights_missing` + `hl_*`.
* **Train/test split:** split by `product_id` (e.g. `GroupShuffleSplit`), never by row.
* **Do not use as predictors:** `rating`, `recommended`, the 2023 flags (`new`, `online_only`, `limited_edition`, `out_of_stock`, `sephora_exclusive`).
* **Part F:** recommendation outcome on observed-response rows and log-price sensitivity. Recent-review and repeat-author subsets are available features but are not part of the final fitted specifications.
* **Bootstrap (Part D):** resample `product_id`s, not rows.

## AI use statement (Part B)

Following the course policy, the main uses of generative AI (Claude) in Part B were:

| Use | What AI did | What the member checked |
|---|---|---|
| Coding | Drafted and debugged the pandas cleaning and feature-engineering code | Ran the notebook end to end; every step ends with `assert` checks |
| Data checks | Suggested checks, e.g. text similarity for repeated reviews, missingness by year, leakage tests | Reviewed the results and the decisions against the proposal |
| Figures | Drafted the matplotlib code for figures B1–B4 | Checked labels and numbers against the tables |
| Documentation | Drafted the cleaning log, feature dictionary and report text | All numbers come from this executed notebook |